In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [32]:
# Systems parameters
a = 0.27
m = 0.08
dw = 1.0
dn = 0.001

In [ ]:
# Find homogeneous equilibrium (W0, N0)
# m*N^2 - a*N + m = 0

coef = [m, -a, m]
roots = np.roots(coef)
roots_real = roots[np.abs(np.imag(roots)) < 1e-10]
roots_real = np.real(roots_real)

print("Real roots of N:")
print(roots_real)

# Choose the smallest positive root for N0
N0 = np.max(roots_real) # Testing the maximum root for stability
W0 = m / N0

print("\nChosen equilibrium:")
print("W0 =", W0)
print("N0 =", N0)

Real roots of N:
[3.0467852 0.3282148]

Chosen equilibrium:
W0 = 0.026257184145342272
N0 = 3.0467851981832217


In [34]:
# Evaluate the Jacobian at the equilibrium
# f(W,N) = a - W - W*N^2
# g(W,N) = W*N^2 - m*N

fW = -1 - N0**2
fN = -2 * W0 * N0
gW = N0**2
gN = 2 * W0 * N0 - m

J = np.array([
    [fW, fN],
    [gW, gN]
])

# Stability without diffusion

eig_no_diff = np.linalg.eigvals(J)

print("\nAutovalores sem difusão:")
print(eig_no_diff)

# Classify stability based on eigenvalues
re = np.real(eig_no_diff)

if np.all(re < 0):
    stability = "Stable equilibrium."
elif np.all(re > 0):
    stability = "Unstable equilibrium."
elif np.any(re < 0) and np.any(re > 0):
    stability = "Saddle point."
else:
    stability = "Marginal case."

print("\nType of stability:")
print(stability)


Autovalores sem difusão:
[-10.13753584  -0.06536421]

Type of stability:
Stable equilibrium.


In [36]:
# Detecting Turing instability
k_vals = np.linspace(0, 20, 600)

growth = []

for k in k_vals:

    # Linearized system with diffusion
    A = np.array([
        [fW - dw * k**2, fN],
        [gW, gN - dn * k**2]
    ])

    eigvals = np.linalg.eigvals(A)
    growth.append(np.max(np.real(eigvals)))

growth = np.array(growth)

# Detecting the maximum growth rate and corresponding wavenumber

max_growth = np.max(growth)
k_max = k_vals[np.argmax(growth)]

print("\nMaximum growth rate:")
print(max_growth)

print("\nWavenumber of the most unstable mode:")
print(k_max)

# Turing instability

lambda_0 = growth[0]
lambda_max = np.max(growth)
k_max = k_vals[np.argmax(growth)]

if lambda_0 < 0 and lambda_max > 0:
    print("\nTuring instability")
    print(f"Most unstable mode: k = {k_max:.4f}")
    print(f"Max growth rate: {lambda_max:.6f}")

elif lambda_0 > 0:
    print("\nHomogeneous instability (NOT Turing)")
    print(f"Max growth at k = {k_max:.4f}")

else:
    print("\nNo instability")

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_vals,
    y=growth,
    mode='lines',
    name='Re(λ_max)'
))

fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    title="Relação de dispersão",
    xaxis_title="k",
    yaxis_title="Re(λ)",
)

fig.show()


Maximum growth rate:
0.013217780800137496

Wavenumber of the most unstable mode:
5.308848080133556

Turing instability
Most unstable mode: k = 5.3088
Max growth rate: 0.013218


#### Code with sliders to variate dn and dw

In [37]:
# Function to calculate the stability analysis for given parameters
def compute_growth(a, m, dw, dn, k_vals):

    # Roots of the homogeneous equilibrium
    coef = [m, -a, m]
    roots = np.roots(coef)
    roots_real = np.real(roots[np.abs(np.imag(roots)) < 1e-10])

    N0 = np.max(roots_real)
    W0 = m / N0

    # Terms of the Jacobian
    fW = -1 - N0**2
    fN = -2 * W0 * N0
    gW = N0**2
    gN = 2 * W0 * N0 - m

    growth = []

    # Loop over wavenumbers to compute the growth rate
    for k in k_vals:
        A = np.array([
            [fW - dw * k**2, fN],
            [gW, gN - dn * k**2]
        ])
        eigvals = np.linalg.eigvals(A)
        growth.append(np.max(np.real(eigvals)))

    return np.array(growth)

In [42]:
k_vals = np.linspace(0, 20, 300)

dn_values = np.linspace(0.0005, 0.02, 50)
dw_values = np.linspace(0.5, 2.0, 50)

a = 0.27
m = 0.08

In [43]:
data = {}

for dn in dn_values:
    for dw in dw_values:
        data[(dn, dw)] = compute_growth(a, m, dw, dn, k_vals)

In [44]:
def get_y(dn, dw):
    return data[(dn, dw)]

In [45]:
fig = go.Figure()

dn0 = dn_values[0]
dw0 = dw_values[0]

fig.add_trace(go.Scatter(
    x=k_vals,
    y=get_y(dn0, dw0),
    mode="lines",
    name="Re(λ_max)"
))

dn_steps = []

for dn in dn_values:
    dn_steps.append(dict(
        method="update",
        args=[
            {"y": [get_y(dn, dw0)]},
            {"title": f"dn = {dn:.4f}, dw = {dw0:.2f}"}
        ],
        label=f"{dn:.4f}"
    ))

dw_buttons = []

for dw in dw_values:
    dw_buttons.append(dict(
        method="update",
        args=[
            {"y": [get_y(dn0, dw)]},
            {"title": f"dn = {dn0:.4f}, dw = {dw:.2f}"}
        ],
        label=f"{dw:.2f}"
    ))

fig.update_layout(
    title="Relação de dispersão (Turing instability)",
    xaxis_title="k",
    yaxis_title="Re(λ_max)",

    sliders=[dict(
        active=0,
        currentvalue={"prefix": "dn = "},
        steps=dn_steps
    )],

    updatemenus=[dict(
        type="dropdown",
        direction="down",
        x=1.0,
        y=1.0,
        showactive=True,
        buttons=dw_buttons
    )]
)

fig.add_hline(y=0, line_dash="dash")

fig.show()